In [2]:
# 라이브러리 설치
!pip install pandas numpy scikit-learn transformers torch datasets

이 셀은 데이터 처리, 머신러닝, 딥러닝 작업에 필요한 pandas, numpy, scikit-learn, transformers, torch, datasets와 같은 Python 라이브러리들을 설치합니다.

In [3]:
# 라이브러리 import
import pandas as pd
import numpy as np

import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score

from transformers import BertTokenizer, BertForSequenceClassification
from transformers import Trainer, TrainingArguments

이 셀은 설치된 라이브러리들을 임포트하여 노트북에서 사용할 수 있도록 준비합니다. 데이터 처리, 모델 구축, 평가 등에 필요한 주요 모듈들을 불러옵니다.

In [4]:
# 구글 드라이브 연결
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


이 셀은 Colab 환경을 Google Drive에 연결하여 Drive에 저장된 파일에 접근할 수 있도록 합니다. 데이터셋을 Drive에서 불러올 때 필요합니다.

In [5]:
# zip 파일 경로 설정
zip_path = "/content/drive/MyDrive/archive.zip"

이 셀은 Google Drive에 저장된 압축 파일(`archive.zip`)의 경로를 `zip_path` 변수에 설정합니다.

In [6]:
# 압축 해제
import zipfile
import os

extract_path = "/content/data"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("압축 해제 완료!")
print(os.listdir(extract_path))

압축 해제 완료!
['True.csv', 'Fake.csv']


이 셀은 `zip_path`에 지정된 압축 파일을 `/content/data` 디렉토리로 해제합니다. 데이터 분석을 위한 원본 파일들을 준비하는 단계입니다.

In [7]:
# CSV 파일 로드
import pandas as pd

fake = pd.read_csv(os.path.join(extract_path, "Fake.csv"))
real = pd.read_csv(os.path.join(extract_path, "True.csv"))

이 셀은 압축 해제된 디렉토리에서 `Fake.csv`와 `True.csv` 파일을 각각 `fake`와 `real`이라는 pandas DataFrame으로 로드합니다. 이는 가짜 뉴스 데이터와 진짜 뉴스 데이터를 구분하여 불러오는 과정입니다.

In [8]:
# 자동 파일 찾기 버전
files = os.listdir(extract_path)

fake_file = [f for f in files if "Fake" in f or "fake" in f][0]
real_file = [f for f in files if "True" in f or "real" in f][0]

fake = pd.read_csv(os.path.join(extract_path, fake_file))
real = pd.read_csv(os.path.join(extract_path, real_file))

print(fake_file, real_file)

Fake.csv True.csv


이 셀은 `Fake.csv`와 `True.csv` 파일을 자동으로 찾아 로드하는 대체 방법입니다. 파일 이름을 직접 지정하는 대신, 디렉토리 내에서 'Fake' 또는 'True'를 포함하는 파일을 찾아 유연하게 데이터를 로드할 수 있도록 합니다.

In [9]:
# 전처리

fake["label"] = 0
real["label"] = 1

df = pd.concat([fake, real])
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# 텍스트 결합
df["text"] = df["title"] + " " + df["text"]
df = df[["text", "label"]]

df.head()

,text,label
0,Ben Stein Calls Out 9th Circuit Court: Committ...,0
1,Trump drops Steve Bannon from National Securit...,1
2,Puerto Rico expects U.S. to lift Jones Act shi...,1
3,OOPS: Trump Just Accidentally Confirmed He Le...,0
4,Donald Trump heads for Scotland to reopen a go...,1


이 셀은 데이터 전처리를 수행합니다. `fake` 데이터에는 레이블 0을, `real` 데이터에는 레이블 1을 할당한 후, 두 데이터를 하나로 합칩니다. 그리고 데이터를 무작위로 섞은 다음, 'title'과 'text' 컬럼을 합쳐 새로운 'text' 컬럼을 만듭니다. 마지막으로 전처리된 데이터의 상위 5개를 출력하여 확인합니다.

In [10]:
# 데이터 분할
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["text"], df["label"], test_size=0.2, random_state=42
)

이 셀은 전처리된 데이터를 훈련 세트와 테스트 세트로 분할합니다. `df['text']`와 `df['label']`을 사용하여 텍스트 데이터와 레이블 데이터를 각각 훈련 및 테스트용으로 나눕니다. `test_size=0.2`는 전체 데이터의 20%를 테스트 세트로 사용함을 의미하며, `random_state=42`는 재현 가능한 분할을 위해 사용됩니다.

In [11]:
# Baseline 모델(Naive Bayes)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

vectorizer = TfidfVectorizer(max_features=5000)

X_train = vectorizer.fit_transform(train_texts)
X_test = vectorizer.transform(test_texts)

nb_model = MultinomialNB()
nb_model.fit(X_train, train_labels)

nb_preds = nb_model.predict(X_test)

print("✅ Naive Bayes 결과")
print(classification_report(test_labels, nb_preds))

✅ Naive Bayes 결과
              precision    recall  f1-score   support

           0       0.93      0.94      0.94      4710
           1       0.93      0.93      0.93      4270

    accuracy                           0.93      8980
   macro avg       0.93      0.93      0.93      8980
weighted avg       0.93      0.93      0.93      8980



이 셀은 TF-IDF(Term Frequency-Inverse Document Frequency) 벡터화를 사용하여 텍스트 데이터를 수치형으로 변환하고, 이를 기반으로 Naive Bayes 모델을 훈련 및 평가합니다. 이는 BERT 모델과 비교할 베이스라인 모델로 사용됩니다. 분류 보고서(`classification_report`)를 출력하여 모델의 성능을 자세히 확인합니다.

In [12]:
# BERT 토크나이저
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

이 셀은 사전 훈련된 BERT 모델(`bert-base-uncased`)의 토크나이저를 로드합니다. 토크나이저는 텍스트를 BERT 모델이 이해할 수 있는 숫자 토큰 시퀀스로 변환하는 역할을 합니다.

In [13]:
# BERT 입력 변환
train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=256
)

test_encodings = tokenizer(
    list(test_texts),
    truncation=True,
    padding=True,
    max_length=256
)

이 셀은 로드된 BERT 토크나이저를 사용하여 훈련 세트와 테스트 세트의 텍스트 데이터를 BERT 모델의 입력 형식에 맞게 변환합니다. `truncation=True`는 최대 길이를 초과하는 텍스트를 자르고, `padding=True`는 길이가 짧은 텍스트를 채우며, `max_length=256`은 입력 시퀀스의 최대 길이를 256으로 설정합니다.

In [14]:
# Bert 모델 로드
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


이 셀은 사전 훈련된 `bert-base-uncased` 모델을 시퀀스 분류 작업에 맞게 로드합니다. `num_labels=2`는 이진 분류(가짜 뉴스/진짜 뉴스)를 수행함을 나타냅니다. 로드 시 출력되는 경고 메시지는 모델 구조 변경으로 인한 것이며, 일반적으로 학습에는 문제가 없습니다.

In [15]:
import torch

# Dataset 클래스 정의
class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = NewsDataset(train_encodings, train_labels)
test_dataset = NewsDataset(test_encodings, test_labels)

이 셀은 PyTorch의 `Dataset` 클래스를 상속받아 `NewsDataset`을 정의합니다. 이 클래스는 BERT 토크나이저를 통해 인코딩된 데이터와 레이블을 묶어 PyTorch 모델이 학습할 수 있는 형태로 제공합니다. 데이터를 효율적으로 로드하고 배치 처리하기 위해 필요합니다.

In [16]:
# 학습 설정
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./logs",
    eval_strategy="epoch", # evaluation_strategy -> eval_strategy로 변경
    save_strategy="epoch",
    load_best_model_at_end=True
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


이 셀은 BERT 모델 훈련에 필요한 `TrainingArguments`를 설정합니다. 여기에는 모델 출력 디렉토리, 훈련 에폭 수, 배치 크기, 로깅 디렉토리, 평가 전략(`epoch`마다 평가), 저장 전략, 그리고 가장 좋은 모델을 로드할지 여부 등이 포함됩니다.

In [17]:
# Trainer 생성
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

이 셀은 Hugging Face `transformers` 라이브러리의 `Trainer` 객체를 생성합니다. 이 `Trainer`는 위에서 정의한 BERT 모델, 훈련 설정(`training_args`), 훈련 데이터셋, 평가 데이터셋을 인자로 받아 모델 훈련 과정을 간소화하고 관리합니다.

In [18]:
# 학습 실행
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.006214,0.008216
2,0.000007,0.004061


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=8980, training_loss=0.007395154041059747, metrics={'train_runtime': 4147.4732, 'train_samples_per_second': 17.32, 'train_steps_per_second': 2.165, 'total_flos': 9450422886420480.0, 'train_loss': 0.007395154041059747, 'epoch': 2.0})

이 셀은 `trainer.train()` 메서드를 호출하여 BERT 모델의 실제 훈련을 시작합니다. 설정된 에폭 수와 배치 크기에 따라 모델이 훈련 데이터셋을 학습하고 가중치를 업데이트합니다.

In [19]:
# 평가
predictions = trainer.predict(test_dataset)
preds = predictions.predictions.argmax(axis=1)

print("✅ BERT 결과")
print(classification_report(test_labels, preds))

print("Accuracy:", accuracy_score(test_labels, preds))
print("F1 Score:", f1_score(test_labels, preds))

✅ BERT 결과
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4710
           1       1.00      1.00      1.00      4270

    accuracy                           1.00      8980
   macro avg       1.00      1.00      1.00      8980
weighted avg       1.00      1.00      1.00      8980

Accuracy: 0.9993318485523385
F1 Score: 0.9992969299273494


이 셀은 훈련된 BERT 모델을 테스트 데이터셋(`test_dataset`)으로 평가합니다. `trainer.predict()`를 사용하여 예측값을 얻고, 이를 실제 레이블과 비교하여 분류 보고서(`classification_report`), 정확도(`accuracy_score`), F1 점수(`f1_score`)를 출력하여 모델의 성능을 자세히 분석합니다.

In [20]:
# 결과 비교
print("===== 모델 비교 =====")
print("Naive Bayes Accuracy:", accuracy_score(test_labels, nb_preds))
print("BERT Accuracy:", accuracy_score(test_labels, preds))

===== 모델 비교 =====
Naive Bayes Accuracy: 0.9320712694877505
BERT Accuracy: 0.9993318485523385


이 셀은 Naive Bayes 베이스라인 모델과 BERT 모델의 최종 정확도를 비교하여 출력합니다. 이를 통해 BERT 모델이 베이스라인 모델 대비 얼마나 성능이 향상되었는지 한눈에 확인할 수 있습니다.

In [21]:
wrong = []

for i in range(len(test_labels)):
    if test_labels.iloc[i] != preds[i]:
        wrong.append((test_texts.iloc[i], test_labels.iloc[i], preds[i]))

# 몇 개 출력
for i in range(5):
    print("TEXT:", wrong[i][0][:200])
    print("TRUE:", wrong[i][1], "PRED:", wrong[i][2])
    print("-"*50)

TEXT: Macedonia's parliament adopts 2018 budget, opposition boycotts vote SKOPJE - Macedonia s parliament has adopted a 2018 draft budget, lowering the deficit to 2.7 percent of national output from 2.9 thi
TRUE: 1 PRED: 0
--------------------------------------------------
TEXT: The Adoration Of Kim Jong Un PYONGYANG - North Koreans stage a demonstration of devotion to their leader Kim Jong Un at least once a year, in a large ceremonial square in Pyongyang.  Mansae!  the peop
TRUE: 1 PRED: 0
--------------------------------------------------
TEXT: Jenner uses women's restroom at Trump property Caitlyn Jenner posted a video on Wednesday (April 26) of herself using a women’s bathroom at Republican Presidential candidate Donald Trump’s New York ho
TRUE: 1 PRED: 0
--------------------------------------------------
TEXT: Commentary: Party leaders often disliked their nominee. It’s the public vitriol that’s new. GOP leaders have unleashed a stunning level of vitriol against their party’s mos

In [22]:
num_misclassified = len(wrong)
print(f"Total misclassified examples: {num_misclassified}개")

Total misclassified examples: 6개


이 셀은 모델이 잘못 분류한 전체 샘플 수를 출력합니다. 이를 통해 모델의 오류율을 정량적으로 파악할 수 있습니다.

이 셀은 BERT 모델이 잘못 분류한 사례들을 분석합니다. `test_labels`와 `preds`를 비교하여 예측이 틀린 경우의 텍스트, 실제 레이블, 예측 레이블을 `wrong` 리스트에 저장합니다. 그리고 처음 5개의 오분류 사례를 출력하여 모델이 어떤 유형의 데이터에서 어려움을 겪는지 시각적으로 파악할 수 있도록 돕습니다. 이 셀이 실행되려면 `test_labels`, `preds`, `test_texts` 변수가 이전 셀들에서 올바르게 정의되어야 합니다.